In [1]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("./data/sekunder/tomato irrigation dataset.csv")
print(f"Data: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")
df_raw.head()

Data: 3000 baris, 14 kolom


,air_temperature,air_humidity,soil_moisture,Reference evapotranspiration,Evapotranspiration,Crop Coefficient,Crop Coefficient stage,nitrogen,fosfor,kalium,Solar Radiation ghi,Wind Speed,plant_age,pH
0,31.2,93.6,567.0,563.000086,236.460036,0.42,Initial Stage,107,38,53,622.0,2.09,1,3.32
1,31.2,93.6,567.0,561.176578,235.694163,0.42,Initial Stage,107,38,53,622.0,2.09,1,3.77
2,30.5,74.6,307.0,561.267170,235.732211,0.42,Initial Stage,107,38,53,622.0,2.09,3,2.90
3,30.4,76.6,308.0,559.447778,234.968067,0.42,Initial Stage,107,38,53,622.0,2.09,3,7.71
4,30.4,76.6,308.0,559.447778,234.968067,0.42,Initial Stage,107,38,53,622.0,2.09,3,5.11


# Cek data

In [2]:
df_raw[["soil_moisture", "air_temperature","air_humidity", "nitrogen", "fosfor", "kalium"]].describe().round(2)

,soil_moisture,air_temperature,air_humidity,nitrogen,fosfor,kalium
count,3000.00,3000.00,3000.00,3000.00,3000.00,3000.00
mean,393.45,25.38,77.59,91.18,71.64,86.87
std,167.86,4.05,9.95,18.08,21.95,29.51
min,120.09,18.00,60.00,44.00,15.00,22.00
25%,240.28,21.96,69.18,76.00,52.00,58.00
50%,386.16,25.65,77.80,92.00,72.00,86.00
75%,539.57,28.90,85.80,107.00,90.00,113.00
max,699.99,32.00,98.20,119.00,109.00,139.00


# Normalisasi 

In [3]:
df = df_raw[["nitrogen", "fosfor", "kalium", "plant_age"]].copy()

sm_min = df_raw["soil_moisture"].min()
sm_max = df_raw["soil_moisture"].max()
df["soil_moisture"] = 100 - ((df_raw["soil_moisture"] - sm_min) / (sm_max - sm_min) * 100)
df["soil_moisture"] = df["soil_moisture"].clip(10, 90).round(2)

print("Fitur berhasil diekstrak dan dinormalisasi!")
df[["nitrogen", "fosfor", "kalium", "soil_moisture", "plant_age"]].describe().round(2)

Fitur berhasil diekstrak dan dinormalisasi!


,nitrogen,fosfor,kalium,soil_moisture,plant_age
count,3000.00,3000.00,3000.00,3000.00,3000.00
mean,91.18,71.64,86.87,52.76,79.77
std,18.08,21.95,29.51,27.47,29.16
min,44.00,15.00,22.00,10.00,1.00
25%,76.00,52.00,58.00,27.66,62.00
50%,92.00,72.00,86.00,54.12,85.00
75%,107.00,90.00,113.00,79.27,100.00
max,119.00,109.00,139.00,90.00,130.00


## Buat fase dari plant age

In [4]:
def get_fase(age):
    if age <= 30:   return 0    # establishment
    elif age <= 55: return 1    # vegetatif
    elif age <= 75: return 2    # berbunga
    else:           return 3    # pematangan

df["fase"] = df["plant_age"].apply(get_fase)

FASE_NAMES = {0: "establishment", 1: "vegetatif", 2: "berbunga", 3: "pematangan"}
df["fase_label"] = df["fase"].map(FASE_NAMES)

print(df.columns.tolist())

['nitrogen', 'fosfor', 'kalium', 'plant_age', 'soil_moisture', 'fase', 'fase_label']


In [5]:
df[["nitrogen", "fosfor", "kalium", "soil_moisture", "plant_age", "fase"]].describe().round(2)

,nitrogen,fosfor,kalium,soil_moisture,plant_age,fase
count,3000.00,3000.00,3000.00,3000.00,3000.00,3000.00
mean,91.18,71.64,86.87,52.76,79.77,2.35
std,18.08,21.95,29.51,27.47,29.16,0.96
min,44.00,15.00,22.00,10.00,1.00,0.00
25%,76.00,52.00,58.00,27.66,62.00,2.00
50%,92.00,72.00,86.00,54.12,85.00,3.00
75%,107.00,90.00,113.00,79.27,100.00,3.00
max,119.00,109.00,139.00,90.00,130.00,3.00


# Rasio Ideal NPK per Fase (dari Haifa)

In [6]:
# Rasio N:P2O5:K2O ideal per fase (Haifa Crop Guide)
RASIO_IDEAL = {
    0: (1, 2, 1),   # establishment - P dominan (root development)
    1: (1, 1, 1),   # vegetatif - seimbang
    2: (2, 1, 3),   # berbunga - K dominan, P turun
    3: (2, 1, 3),   # pematangan - K dominan
}

# Batas absolut per unsur (mg/kg) - sesuaikan skala sensormu
BATAS_MIN = {"n": 40, "p": 50, "k": 60}
BATAS_MAX = {"n": 150, "p": 150, "k": 200}

print("Rasio ideal & batas siap.")

Rasio ideal & batas siap.


# Labelling dataset pupuk

In [7]:
LABEL_NAMES = {
    0: "Tidak perlu",
    1: "Urea/ZA",              # N kurang
    2: "SP-36",                # P kurang
    3: "KCl",                  # K kurang
    4: "Urea/ZA + SP-36",      # N,P kurang
    5: "Urea/ZA + KCl",        # N,K kurang
    6: "SP-36 + KCl",          # P,K kurang
    7: "Urea/ZA + SP-36 + KCl",# semua kurang
    8: "NPK 15-15-15",         # maintenance
    9: "Kurangi pemupukan N",  # N berlebih
    10: "Flush air (nutrisi tinggi)",  # over (Nama disesuaikan, EC dibuang)
}

def label_pupuk(row):
    n, p, k = row["nitrogen"], row["fosfor"], row["kalium"]
    fase = row["fase"] 

    # Cek kalau N dan K over limit
    if n > BATAS_MAX["n"] and k > BATAS_MAX["k"]: return 10
    if n > BATAS_MAX["n"]: return 9

    rn, rp, rk = RASIO_IDEAL[fase]  
    total_ratio = rn + rp + rk
    total_npk = n + p + k
    
    if total_npk == 0: return 7

    prop_n, prop_p, prop_k = n/total_npk, p/total_npk, k/total_npk
    ideal_n, ideal_p, ideal_k = rn/total_ratio, rp/total_ratio, rk/total_ratio

    TOLERANSI = 0.7
    n_low = (prop_n < ideal_n * TOLERANSI) or (n < BATAS_MIN["n"])
    p_low = (prop_p < ideal_p * TOLERANSI) or (p < BATAS_MIN["p"])
    k_low = (prop_k < ideal_k * TOLERANSI) or (k < BATAS_MIN["k"])

    if   n_low and p_low and k_low: return 7
    elif n_low and p_low:           return 4
    elif n_low and k_low:           return 5
    elif p_low and k_low:           return 6
    elif n_low:                     return 1
    elif p_low:                     return 2
    elif k_low:                     return 3
    else:                           return 0

print("Fungsi label_pupuk() siap.")

Fungsi label_pupuk() siap.


In [8]:
df["recommendation"] = df.apply(label_pupuk, axis=1)
df["recommendation_label"] = df["recommendation"].map(LABEL_NAMES)

print("Distribusi rekomendasi pupuk:")
dist = df["recommendation"].value_counts().sort_index()
for code, count in dist.items():
    print(f"  {code} = {LABEL_NAMES[code]:35s} {count:5d} ({count/len(df)*100:.1f}%)")

Distribusi rekomendasi pupuk:
  0 = Tidak perlu                          1129 (37.6%)
  1 = Urea/ZA                                47 (1.6%)
  2 = SP-36                                 301 (10.0%)
  3 = KCl                                  1133 (37.8%)
  6 = SP-36 + KCl                           390 (13.0%)


# Preview Hasil

In [9]:
df[["plant_age", "fase_label", "nitrogen", "fosfor", "kalium", "soil_moisture", "recommendation_label"]].head(20)

,plant_age,fase_label,nitrogen,fosfor,kalium,soil_moisture,recommendation_label
0,1,establishment,107,38,53,22.93,SP-36 + KCl
1,1,establishment,107,38,53,22.93,SP-36 + KCl
2,3,establishment,107,38,53,67.77,SP-36 + KCl
3,3,establishment,107,38,53,67.60,SP-36 + KCl
4,3,establishment,107,38,53,67.60,SP-36 + KCl
5,3,establishment,107,38,53,67.77,SP-36 + KCl
6,3,establishment,107,38,53,67.77,SP-36 + KCl
7,3,establishment,107,38,53,67.77,SP-36 + KCl
8,3,establishment,107,38,53,67.77,SP-36 + KCl
9,3,establishment,107,38,53,67.77,SP-36 + KCl


# Simpan dataset

In [10]:
cols_pupuk = ["nitrogen", "fosfor", "kalium", "plant_age", "fase", "soil_moisture", "recommendation"]

df[cols_pupuk].to_csv("data/dataset_pupuk_final.csv", index=False)

print("Tersimpan: dataset_pupuk_final.csv")
print(f"Total: {len(df)} baris siap masuk Random Forest!")

Tersimpan: dataset_pupuk_final.csv
Total: 3000 baris siap masuk Random Forest!
